# 10 — Extensions

Asymmetric goal amplitudes, three goals, and parametric $\delta$ forcing. (Noise / Kramers escape and slow $\delta$ dynamics are sketched but not fully developed.)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from steering.params import ModelParams, ForcingParams
from steering.models import (
    DuffingModel,
    BesselSteeringModel,
    ContinuousPFLModel,
    DiscretePFLModel,
    FullCircuitModel,
)
from steering.dynamics import VelocityDynamics, AccelerationDynamics
from steering.integrator import Simulation
from steering.visualization.style import use_paper_style

use_paper_style()


## Asymmetric amplitudes

Build a thin override of the continuous model with $A_1 \neq A_2$ on the two goal bumps. The pitchfork unfolds into an imperfect bifurcation.

In [ ]:
from steering.nonlinearities import get_nonlinearity
from scipy.integrate import fixed_quad

class AsymmetricContinuous(ContinuousPFLModel):
    def __init__(self, A1=1.0, A2=1.0, n_quad=256):
        super().__init__(n_quad=n_quad)
        self.A1 = A1; self.A2 = A2

    def _population_sum(self, theta, params, shift):
        f = get_nonlinearity(params.nonlinearity, params.nonlinearity_params)
        theta_arr = np.atleast_1d(np.asarray(theta, dtype=float))
        phi = self._phi_nodes[None, :]; theta_b = theta_arr[:, None]
        h = (params.S * np.exp(params.kappa_h * np.cos(theta_b - phi + shift))
             + self.A1 * np.exp(params.kappa_g * np.cos(params.delta - phi))
             + self.A2 * np.exp(params.kappa_g * np.cos(-params.delta - phi)))
        fh = f(h)
        return np.einsum('ij,j->i', fh, self._phi_weights)

theta = np.linspace(-np.pi, np.pi, 401)
p = ModelParams(kappa_h=2.0, kappa_g=2.0, delta=1.4)
fig, ax = plt.subplots(figsize=(7, 4))
for r in [1.0, 0.9, 0.7, 0.5]:
    m = AsymmetricContinuous(A1=1.0, A2=r)
    U = m.steering_drive(theta, p)
    ax.plot(theta, U, label=fr'$A_2/A_1={r}$')
ax.axhline(0, color='0.6', lw=0.5)
ax.set_xlabel(r'$\theta$'); ax.set_ylabel(r'$U(\theta)$')
ax.legend(); plt.show()


## Parametric $\delta(t)$ forcing

In [ ]:
bessel = BesselSteeringModel()
p = ModelParams(kappa_h=2.0, kappa_g=2.0, delta=1.4)
dyn = AccelerationDynamics(model=bessel, gamma=0.1)
forc = ForcingParams(F=0.3, omega=2.0, forcing_type='parametric_delta')
sim = Simulation(dyn, p, forc, rtol=1e-9, atol=1e-11)
res = sim.run(np.array([0.5, 0.0]), (0.0, 80.0))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(res.t, res.states[:, 0])
ax.set_xlabel('t'); ax.set_ylabel(r'$\theta$')
ax.set_title(r'Parametric $\delta(t)$ forcing')
plt.show()
